# FFT tutorial

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/imaging/astroviper_tutorial_fft.ipynb)

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper[all]")

    import astroviper  # noqa: F401 -- availability probe for the pip-install fallback

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

In [ ]:
# from graphviper.dask.client import local_client
# viper_client = local_client(cores=4, memory_limit="4GB")

import dask

dask.config.set(scheduler="synchronous")

In [ ]:
import numpy as np
import xarray as xr
from xradio.image.image import make_empty_sky_image

arcsec_to_rad = np.pi / 180 / 3600

im_xds = make_empty_sky_image(
    phase_center=[0.6, -0.2],
    image_size=[1024, 1024],
    cell_size=[arcsec_to_rad, arcsec_to_rad],
    frequency_coords=np.linspace(1.0e11, 1.1e11, 16),
    pol_coords=["I"],
    time_coords=[0],
)

im_xds

In [ ]:
import dask.array as da

sky = da.random.random_sample(
    [1, 16, 1024, 1024, 1], chunks=[1, 2, 1024, 1024, 1]
).astype("float32")

xda = xr.DataArray(sky, dims=["time", "frequency", "l", "m", "polarization"])
im_xds["SKY"] = xda
input_data = {"img": im_xds}
im_xds

In [ ]:
import os
import shutil

from xradio.image import write_image

zarr_name = "test.zarr"
if os.path.exists(zarr_name):
    shutil.rmtree(zarr_name)
write_image(im_xds, zarr_name, "zarr")

In [ ]:
from graphviper.graph_tools.coordinate_utils import make_parallel_coord
from IPython.display import HTML, display
from toolviper.utils.display import dict_to_html

parallel_coords = {}
n_chunks = 4
parallel_coords["frequency"] = make_parallel_coord(
    coord=im_xds.frequency, n_chunks=n_chunks
)
display(HTML(dict_to_html(parallel_coords["frequency"])))

In [ ]:
from graphviper.graph_tools.coordinate_utils import (
    interpolate_data_coords_onto_parallel_coords,
)

node_task_data_mapping = interpolate_data_coords_onto_parallel_coords(
    parallel_coords, input_data
)
display(HTML(dict_to_html(node_task_data_mapping)))

In [ ]:
import dask
from graphviper.graph_tools.map import map


def _fft2(input_params):
    display(HTML(dict_to_html(input_params)))

    from xradio.image import load_image

    if input_params["input_data"] is None:  # Load
        img_xds = load_image(
            input_params["input_data_store"],
            block_des=input_params["data_selection"]["img"],
        )
    else:
        img_xds = input_params["input_data"]["img"]  # In memory

    display(img_xds)

    fft_plane = (
        img_xds["SKY"].dims.index(input_params["axes"][0]),
        img_xds["SKY"].dims.index(input_params["axes"][1]),
    )
    print("fft_plane", fft_plane)
    aperture = np.fft.fftshift(
        np.fft.fft2(np.fft.ifftshift(img_xds.SKY, axes=fft_plane), axes=fft_plane),
        axes=fft_plane,
    ).real

    img_xds["APERTURE"] = xr.DataArray(
        aperture, dims=("time", "frequency", "u", "v", "polarization")
    )

    return img_xds


input_params = {}
input_params["input_data_store"] = zarr_name
input_params["axes"] = ("l", "m")  # (3,4)

viper_graph = map(
    input_data=input_data,
    node_task_data_mapping=node_task_data_mapping,
    node_task=_fft2,
    input_params=input_params,
    in_memory_compute=False,
)
from graphviper.graph_tools import generate_dask_workflow

dask_graph = generate_dask_workflow(viper_graph)
dask.visualize(dask_graph, filename="map_graph")

In [ ]:
aperture_list = dask.compute(dask_graph)

In [ ]:
len(aperture_list[0])

In [ ]:
aperture_list[0][0]

In [ ]:
aperture_list[0][0]["SKY"].sel(polarization="I").isel(frequency=2, time=0).plot()

In [ ]:
aperture_list[0][0]["APERTURE"].sel(polarization="I").isel(frequency=2, time=0).plot(
    vmin=-500, vmax=500
)